# CellMatch on NYU Cloud Bursting — Run All → `submission.csv`

**Run > Run All Cells.**

Works in two modes:
1. **GPU Jupyter** (best): runs training in this session.
2. **Any Jupyter / login shell** (this CPU session is fine): cell submits Slurm jobs on `g2-standard-12` (L4) or `c12m85-a100-1` (A100) and waits until `submission.csv` appears.

Account: `cs_gy_6923-2026fa`. Project: `/scratch/ts5789/cellmatch`.


In [ ]:
from pathlib import Path
import os

WORK = Path("/scratch/ts5789/cellmatch")
assert WORK.exists(), f"Missing {WORK} — clone Cell-Matching there first"
os.chdir(WORK)
os.environ["SLURM_SUBMIT_DIR"] = str(WORK)
os.environ["WORK_DIR"] = str(WORK)
os.environ["CELLPOSE_LOCAL_MODELS_PATH"] = str(WORK / "cellpose_weights")
os.environ["PIP_CACHE_DIR"] = str(WORK / ".pip_cache")
for p in ["logs", "cellpose_weights", ".pip_cache", "models", "cache"]:
    (WORK / p).mkdir(exist_ok=True)
print("WORK =", WORK.resolve())
ACCOUNT = "cs_gy_6923-2026fa"
# Change to c12m85-a100-1 if you want A100 instead of L4
GPU_PARTITION = "g2-standard-12"
GPU_MEM = "40G"
CPU_PARTITION = "n2c48m24"
CPU_MEM = "20G"


In [ ]:
# Ensure project env exists (used by batch jobs). Safe to re-run.
import os, subprocess, time
from pathlib import Path

WORK = Path("/scratch/ts5789/cellmatch")
CONDA, ENV = WORK / "miniforge", WORK / "env"
PY = ENV / "bin" / "python"

def run(cmd):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.check_call(cmd)

if not PY.exists():
    print("Bootstrapping env on scratch (one-time, several minutes)...")
    if not (CONDA / "bin" / "conda").exists():
        sh = WORK / "miniforge.sh"
        run(["curl", "-fsSL", "-o", str(sh),
             "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"])
        run(["bash", str(sh), "-b", "-p", str(CONDA)])
        sh.unlink(missing_ok=True)
    run([str(CONDA / "bin" / "conda"), "create", "-y", "-p", str(ENV), "python=3.12"])
    run([str(PY), "-m", "pip", "install", "--upgrade", "pip"])
    run([str(PY), "-m", "pip", "install",
         "torch", "torchvision", "tifffile", "cellpose>=4.2",
         "scikit-image", "scipy", "pandas", "opencv-python-headless"])
print("python:", PY, "exists=", PY.exists())
assert (WORK / "Project_2_Dataset" / "training" / "train_ground_truth.csv").exists(), "dataset missing"
print("dataset OK")


In [ ]:
# If this session already has a GPU, run here. Otherwise submit Slurm chain and wait.
import os, subprocess, time, shutil
from pathlib import Path

WORK = Path("/scratch/ts5789/cellmatch")
PY = WORK / "env" / "bin" / "python"
SUB = WORK / "submission.csv"
ACCOUNT = "cs_gy_6923-2026fa"
GPU_PARTITION = "g2-standard-12"
GPU_MEM = "40G"
CPU_PARTITION = "n2c48m24"
CPU_MEM = "20G"

def cuda_ok():
    try:
        out = subprocess.check_output(
            [str(PY), "-c", "import torch; print(torch.cuda.is_available())"], text=True).strip()
        return out == "True"
    except Exception:
        return False

def sbatch(extra, script):
    # Clear nested-Slurm vars so OOD Jupyter can submit cleanly
    env = {k: v for k, v in os.environ.items() if not k.startswith("SLURM_")}
    env["PATH"] = os.environ.get("PATH", "")
    cmd = [
        "/opt/slurm/bin/sbatch", "--parsable",
        "-A", ACCOUNT, "--export=NONE",
        *extra, script,
    ]
    print("+", " ".join(cmd), flush=True)
    return subprocess.check_output(cmd, text=True, cwd=str(WORK), env=env).strip()

def wait_job(jobid, label):
    while True:
        state = subprocess.check_output(
            ["squeue", "-h", "-j", jobid, "-o", "%T"], text=True, stderr=subprocess.DEVNULL).strip()
        if not state:
            # left queue — check sacct
            acct = subprocess.check_output(
                ["sacct", "-j", jobid, "-n", "-X", "-o", "State"], text=True).strip().split()[0]
            print(label, jobid, "->", acct, flush=True)
            if acct.startswith("COMPLETED"):
                return
            raise RuntimeError(f"{label} job {jobid} ended as {acct}. See logs/")
        print(label, jobid, state, flush=True)
        time.sleep(60)

if cuda_ok():
    print("GPU available in this session — running locally")
    subprocess.check_call([str(PY), "run_hpc.py", "gpu"], cwd=str(WORK))
    subprocess.check_call([str(PY), "run_hpc.py", "cpu"], cwd=str(WORK))
else:
    print("No GPU in this Jupyter session — submitting Slurm jobs")
    print(f"GPU partition={GPU_PARTITION}  (edit cell above for A100: c12m85-a100-1)")
    # Setup only if env somehow incomplete
    if not PY.exists():
        setup = sbatch(["-p", CPU_PARTITION, f"--mem=16G"], "hpc/setup.sbatch")
        wait_job(setup, "setup")
    gpu = sbatch(["-p", GPU_PARTITION, f"--mem={GPU_MEM}", "--gres=gpu:1"], "hpc/gpu.sbatch")
    cpu = sbatch(["-p", CPU_PARTITION, f"--mem={CPU_MEM}", f"--dependency=afterok:{gpu}"], "hpc/cpu.sbatch")
    print("submitted gpu=", gpu, "cpu=", cpu)
    wait_job(gpu, "gpu")
    wait_job(cpu, "cpu")

assert SUB.exists(), "submission.csv missing after jobs"
subprocess.check_call([str(PY), "validate_submission.py", str(SUB)], cwd=str(WORK))
print("DONE →", SUB, f"({SUB.stat().st_size:,} bytes)")
print("\n".join(SUB.read_text().splitlines()[:3]))
